**Release-package note.** The private workstation bootstrap cell was removed. Paths are resolved by `_defs/_init.py`. Before manual use, run `./run prepare-data`; see `README.md` for the tested commands.

In [ ]:
from _defs.iplot import *

In [ ]:
def fit_cos_amplitude_along_dim(coef_sensory):
    """
    对coef_sensory沿指定维度拟合cos函数，返回每个神经元和rank的拟合幅值A
    参数:
        coef_sensory: shape [neuron_num, rank_num, input_num] (或更高阶，只要最后一维为需要拟合的点)
        dim: 沿哪个维度做拟合（默认最后一维）
    返回:
        A_sensory: shape与coef_sensory去除dim后一样
    """
    import numpy as np
    from scipy.optimize import curve_fit
    
    neuron_num = coef_sensory.shape[0]
    rank_num = coef_sensory.shape[1]
    input_num = coef_sensory.shape[2]
    assert input_num == 6

    A_sensory = np.zeros((neuron_num, rank_num))
    theta_sensory = np.zeros((neuron_num, rank_num))
    bias_sensory = np.zeros((neuron_num, rank_num))

    # Define the model function
    def tuning_curve_func(x, a, b, c):
        return a * np.cos(x + b) + c

    for neuron in range(neuron_num):
        for rank in range(rank_num):
            x = np.linspace(0, 2 * np.pi - np.pi / 3, 6)
            y = coef_sensory[neuron, rank, :]
            # Initial guesses for the parameters
            estimated_offset = np.mean(y)
            estimated_A = (np.max(y) - np.min(y)) / 2
            index = np.argmax(y)
            initial_guess = [estimated_A, 2 * np.pi - (index * np.pi / 3), estimated_offset]

            # Bounds for the parameters
            lower_bounds = [0, 0, -np.inf]
            upper_bounds = [np.inf, 2 * np.pi, np.inf]

            # Fit the model to the data
            popt, pcov = curve_fit(tuning_curve_func, x, y, p0=initial_guess, bounds=(lower_bounds, upper_bounds))
            # Store the fitted parameters
            A_sensory[neuron, rank] = popt[0]
            theta_sensory[neuron, rank] = popt[1]
            bias_sensory[neuron, rank] = popt[2]

    return A_sensory

In [ ]:
from sklearn.mixture import GaussianMixture

redrat = 3; fs=6*redrat
fig=plt.figure(figsize=[3.6*redrat,3*redrat])


## model
savepath = outputdir+'/control_location_E_model.npy'
dat = np.load(savepath)
dat = fit_cos_amplitude_along_dim(dat)
x = dat[:,0]; y = dat[:,1]
data = (x-y)/(x+y)

plt.subplot(3,3,1)

plt.scatter(x[1000:], y[1000:], 
            c=colors[0], s=6, label='WM1')
plt.scatter(x[:1000], y[:1000], 
            c=colors[4], s=6, label='WM2')

plt.ylabel('Model\nWM-2 prefered (g2)',fontsize=fs)
plt.ylim([-0.5,5]);plt.xlim([-0.5,5])
plt.xticks([0,2,4],fontsize=fs);plt.yticks([0,2,4],fontsize=fs)
plt.legend(frameon=False,fontsize=fs-4,loc='upper right',bbox_to_anchor=(1.1, 1))
ax1 = plt.gca()  # 獲取當前軸

# 方法1：左上角（最推薦，論文常見位置）
ax1.text(
    0.08, 0.97,          # 相對坐標：x=3%, y=97%（從左下角算起）
    'n = '+str(len(data)),
    transform=ax1.transAxes,  # 使用軸相對坐標（0~1）
    fontsize=fs,              # 調整大小，根據你的 fs 比例
    # fontweight='bold',        # 可選：加粗
    verticalalignment='top',
    horizontalalignment='left',
    bbox=dict(facecolor='white', alpha=0.8, edgecolor='none')  # 可選：半透明白底框
)

plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['bottom'].set_visible(True)
plt.gca().spines['left'].set_visible(True)

plt.subplot(3,3,2)

X = data.reshape(-1, 1)
gmm = GaussianMixture(n_components=2, random_state=42)
gmm.fit(X)

# plt.figure(figsize=(10, 6))
sns.histplot(data, color='gray',bins=100, kde=False, stat='density', alpha=0.2,edgecolor='none')

x = np.linspace(data.min(), data.max(), 1000).reshape(-1, 1)
logprob = gmm.score_samples(x)
pdf = np.exp(logprob)
pdf_individual = gmm.predict_proba(x) * pdf[:, np.newaxis] * gmm.weights_

colors2=['k',colors[4],colors[0]]
# plt.plot(x, pdf, '-k', lw=2)
plt.plot(x, pdf_individual[:, 1], '-', label=f'WM{1}',color=colors2[2],lw=3)
plt.plot(x, pdf_individual[:, 0], '-', label=f'WM{2}',color=colors2[1],lw=3)


plt.ylabel('Density',fontsize=fs)
plt.xticks([-1,0,1],fontsize=fs);plt.yticks([0,5],fontsize=fs)
plt.legend(frameon=False,fontsize=fs-4,loc='upper right',bbox_to_anchor=(1.1, 1))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['bottom'].set_visible(True)
plt.gca().spines['left'].set_visible(True)

plt.subplot(3, 3, 3)
X = data.reshape(-1, 1)
bics = []
for k in range(1, 7):                    # 通常看到6個就夠了
    gmm_temp = GaussianMixture(n_components=k, random_state=42)
    gmm_temp.fit(X)
    bics.append(gmm_temp.bic(X))

plt.plot(range(1, len(bics)+1), bics, 'o-', color='black', lw=2, markersize=8)
# plt.xlabel('Number of Cluster', fontsize=fs)
plt.ylabel('BIC', fontsize=fs)
plt.xticks(range(1, len(bics)+1), fontsize=fs-2)
plt.yticks(fontsize=fs-2)
# plt.grid(True, alpha=0.3)

# 標註最低 BIC 的點
best_k = np.argmin(bics) + 1
plt.plot(best_k, bics[best_k-1], 'o', color='red', markersize=10)
# plt.legend(frameon=False, fontsize=fs-4)

plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['bottom'].set_visible(True)
plt.gca().spines['left'].set_visible(True)
# 2. 再跑一次更稳健的正态检验（推荐）
print("Shapiro p =", stats.shapiro(data).pvalue)
print("KS p     =", stats.kstest(data, lambda x: stats.norm.cdf(x, loc=data.mean(), scale=data.std())).pvalue)


X = data.reshape(-1,1)
bics = []
for k in range(1,6):
    gmm = GaussianMixture(n_components=k, random_state=42)
    gmm.fit(X)
    bics.append(gmm.bic(X))
    
print("BIC (越小越好):", bics)
print("最佳分量数（最低BIC）:", np.argmin(bics)+1)


## swm task
savepath = outputdir+'/control_location_E_raw.pkl'
savedat = load_dictionary_pickle(savepath)
E10 = savedat['E10']; E20 = savedat['E20']

dat = np.array([E10.T,E20.T])
dat = fit_cos_amplitude_along_dim(dat)
x = dat[0,:]; y = dat[1,:]
data = (x-y)/(x+y)

plt.subplot(3,3,4)

X = data.reshape(-1, 1)
gmm = GaussianMixture(n_components=2, random_state=42)
gmm.fit(X)

labels = gmm.predict(X)
plt.scatter(x[labels == 0], y[labels == 0], 
            c=colors[0], s=6)
plt.scatter(x[labels == 1], y[labels == 1], 
            c=colors[4], s=6)

# plt.scatter(x,y,c='gray',s=6)
# plt.xlabel('WM-1 prefered (g1)',fontsize=fs)
plt.ylabel('Monkey\nSWM task\nWM-2 prefered (g2)',fontsize=fs)
plt.ylim([-0.5,7]);plt.xlim([-0.5,7])
plt.xticks([0,3,6],fontsize=fs);plt.yticks([0,3,6],fontsize=fs)
ax1 = plt.gca()  # 獲取當前軸

ax1.text(
    0.08, 0.97,          # 相對坐標：x=3%, y=97%（從左下角算起）
    'n = '+str(len(data)),
    transform=ax1.transAxes,  # 使用軸相對坐標（0~1）
    fontsize=fs,              # 調整大小，根據你的 fs 比例
    # fontweight='bold',        # 可選：加粗
    verticalalignment='top',
    horizontalalignment='left',
    bbox=dict(facecolor='white', alpha=0.8, edgecolor='none')  # 可選：半透明白底框
)

plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['bottom'].set_visible(True)
plt.gca().spines['left'].set_visible(True)

plt.subplot(3,3,5)
X = data.reshape(-1, 1)
gmm = GaussianMixture(n_components=2, random_state=42)
gmm.fit(X)

# plt.figure(figsize=(10, 6))
sns.histplot(data, color='gray',bins=100, kde=False, stat='density', alpha=0.2,edgecolor='none')

x = np.linspace(data.min(), data.max(), 1000).reshape(-1, 1)
logprob = gmm.score_samples(x)
pdf = np.exp(logprob)
pdf_individual = gmm.predict_proba(x) * pdf[:, np.newaxis] * gmm.weights_

colors2=['k',colors[0],colors[4]]
# plt.plot(x, pdf, '-k', lw=2)
for i in range(2):
    plt.plot(x, pdf_individual[:, i], '-', label=f'WM{i+1}',color=colors2[i+1],lw=3)
    # if i==0:
    #     plt.plot(x, pdf_individual[:, i], '-', label=f'Entry',color=colors2[i],lw=3)
    # else:
    #     plt.plot(x, pdf_individual[:, i], '-', label=f'WM{i}',color=colors2[i],lw=3)
# plt.xlabel('(g1-g2)/(g1+g2)',fontsize=fs)
plt.ylabel('Density',fontsize=fs)
plt.xticks([-1,0,1],fontsize=fs);plt.yticks([0,1],fontsize=fs)
# plt.legend(frameon=False,fontsize=fs-4)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['bottom'].set_visible(True)
plt.gca().spines['left'].set_visible(True)

plt.subplot(3, 3, 6)
X = data.reshape(-1, 1)
bics = []
for k in range(1, 7):                    # 通常看到6個就夠了
    gmm_temp = GaussianMixture(n_components=k, random_state=42)
    gmm_temp.fit(X)
    bics.append(gmm_temp.bic(X))

plt.plot(range(1, len(bics)+1), bics, 'o-', color='black', lw=2, markersize=8)
# plt.xlabel('Number of Cluster', fontsize=fs)
plt.ylabel('BIC', fontsize=fs)
plt.xticks(range(1, len(bics)+1), fontsize=fs-2)
plt.yticks(fontsize=fs-2)
# plt.grid(True, alpha=0.3)

# 標註最低 BIC 的點
best_k = np.argmin(bics) + 1
plt.plot(best_k, bics[best_k-1], 'o', color='red', markersize=10)
# plt.legend(frameon=False, fontsize=fs-4)

plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['bottom'].set_visible(True)
plt.gca().spines['left'].set_visible(True)

# 2. 再跑一次更稳健的正态检验（推荐）
print("Shapiro p =", stats.shapiro(data).pvalue)
print("KS p     =", stats.kstest(data, lambda x: stats.norm.cdf(x, loc=data.mean(), scale=data.std())).pvalue)

# 3. 如果还怀疑混合高斯，可以快速试sklearn的GMM（2~4个分量）

X = data.reshape(-1,1)
bics = []
for k in range(1,6):
    gmm = GaussianMixture(n_components=k, random_state=42)
    gmm.fit(X)
    bics.append(gmm.bic(X))
    
print("BIC (越小越好):", bics)
print("最佳分量数（最低BIC）:", np.argmin(bics)+1)






## before learning
savepath = outputdir+'/control_location_E_befrlearn.pkl'
savedat = load_dictionary_pickle(savepath)
E10 = savedat['E10']; E20 = savedat['E20']

dat = np.array([E10.T,E20.T])
dat = fit_cos_amplitude_along_dim(dat)
x = dat[0,:]; y = dat[1,:]
data = (x-y)/(x+y)

plt.subplot(3,3,7)

X = data.reshape(-1, 1)
gmm = GaussianMixture(n_components=2, random_state=42)
gmm.fit(X)
labels = gmm.predict(X)                     # shape: (n_samples,)

plt.scatter(x[labels == 0], y[labels == 0], 
            c='gray', s=6)
plt.scatter(x[labels == 1], y[labels == 1], 
            c='gray', s=6)

plt.xlabel('WM-1 prefered (g1)',fontsize=fs)
plt.ylabel('Monkey \nBefore learning\nWM-2 prefered (g2)',fontsize=fs)
plt.ylim([-0.2,1.5]);plt.xlim([-0.2,1.5])
plt.xticks([0,1],fontsize=fs);plt.yticks([0,1],fontsize=fs)
ax1 = plt.gca()  # 獲取當前軸

# 方法1：左上角（最推薦，論文常見位置）
ax1.text(
    0.08, 0.97,          # 相對坐標：x=3%, y=97%（從左下角算起）
    'n = '+str(len(data)),
    transform=ax1.transAxes,  # 使用軸相對坐標（0~1）
    fontsize=fs,              # 調整大小，根據你的 fs 比例
    # fontweight='bold',        # 可選：加粗
    verticalalignment='top',
    horizontalalignment='left',
    bbox=dict(facecolor='white', alpha=0.8, edgecolor='none')  # 可選：半透明白底框
)

plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['bottom'].set_visible(True)
plt.gca().spines['left'].set_visible(True)

plt.subplot(3,3,8)
# sns.histplot(data, bins=40, kde=True, stat='density')
# plt.title("直方图 + KDE")

X = data.reshape(-1, 1)
gmm = GaussianMixture(n_components=1, random_state=42)
gmm.fit(X)

# plt.figure(figsize=(10, 6))
sns.histplot(data, color='gray',bins=100, kde=False, stat='density', alpha=0.2,edgecolor='none')

x = np.linspace(data.min(), data.max(), 1000).reshape(-1, 1)
logprob = gmm.score_samples(x)
pdf = np.exp(logprob)
pdf_individual = gmm.predict_proba(x) * pdf[:, np.newaxis] * gmm.weights_

colors2=['k',colors[0],colors[4]]
# plt.plot(x, pdf, '-k', lw=2)
for i in range(1):
    plt.plot(x, pdf_individual[:, i], '-', label=f'WM{i+1}',color='gray',lw=3)
    # if i==0:
    #     plt.plot(x, pdf_individual[:, i], '-', label=f'Entry',color=colors2[i],lw=3)
    # else:
    #     plt.plot(x, pdf_individual[:, i], '-', label=f'WM{i}',color=colors2[i],lw=3)
plt.xlabel('(g1-g2)/(g1+g2)',fontsize=fs)
plt.ylabel('Density',fontsize=fs)
plt.xticks([-1,0,1],fontsize=fs);plt.yticks([0,1],fontsize=fs)
# plt.legend(frameon=False,fontsize=fs-4)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['bottom'].set_visible(True)
plt.gca().spines['left'].set_visible(True)


plt.subplot(3, 3, 9)
X = data.reshape(-1, 1)
bics = []
for k in range(1, 7):                    # 通常看到6個就夠了
    gmm_temp = GaussianMixture(n_components=k, random_state=42)
    gmm_temp.fit(X)
    bics.append(gmm_temp.bic(X))

plt.plot(range(1, len(bics)+1), bics, 'o-', color='black', lw=2, markersize=8)
plt.xlabel('Number of Cluster', fontsize=fs)
plt.ylabel('BIC', fontsize=fs)
plt.xticks(range(1, len(bics)+1), fontsize=fs-2)
plt.yticks(fontsize=fs-2)
# plt.grid(True, alpha=0.3)

# 標註最低 BIC 的點
best_k = np.argmin(bics) + 1
plt.plot(best_k, bics[best_k-1], 'o', color='red', markersize=10)
# plt.legend(frameon=False, fontsize=fs-4)

plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['bottom'].set_visible(True)
plt.gca().spines['left'].set_visible(True)

# 2. 再跑一次更稳健的正态检验（推荐）
print("Shapiro p =", stats.shapiro(data).pvalue)
print("KS p     =", stats.kstest(data, lambda x: stats.norm.cdf(x, loc=data.mean(), scale=data.std())).pvalue)

# 3. 如果还怀疑混合高斯，可以快速试sklearn的GMM（2~4个分量）

X = data.reshape(-1,1)
bics = []
for k in range(1,6):
    gmm = GaussianMixture(n_components=k, random_state=42)
    gmm.fit(X)
    bics.append(gmm.bic(X))
    
print("BIC (越小越好):", bics)
print("最佳分量数（最低BIC）:", np.argmin(bics)+1)



plt.tight_layout()
plt.show()

# fig.savefig('/mnt/data2/TP_CON/group/PFC/analysis/05_analysis/data/_figs/'+'fig6g.pdf', format='pdf', dpi=300, bbox_inches='tight')

In [ ]:
redrat = 3
fs = 5 * redrat

# ====================== 创建上下排列的大图 ======================
fig = plt.figure(figsize=[2.2 * redrat, 2.1 * redrat])   # 宽度稍宽一点，高度增加

# ====================== 上半部分：GMM 四个子图 ======================
# before learning
savepath = outputdir + '/control_location_E_befrlearn.pkl'
savedat = load_dictionary_pickle(savepath)
E10 = savedat['E10']
E20 = savedat['E20']
dat = np.array([E10.T, E20.T])
dat = fit_cos_amplitude_along_dim(dat)
x = dat[0, :]
y = dat[1, :]
data = (x - y) / (x + y)
X = data.reshape(-1, 1)

gmm = GaussianMixture(n_components=1, random_state=42)
gmm.fit(X)

plt.subplot(2, 4, 1)
sns.histplot(data, color='gray', bins=50, kde=False, stat='density', alpha=0.2, edgecolor='none')

x_plot = np.linspace(data.min(), data.max(), 1000).reshape(-1, 1)
logprob = gmm.score_samples(x_plot)
pdf = np.exp(logprob)
pdf_individual = gmm.predict_proba(x_plot) * pdf[:, np.newaxis] * gmm.weights_

for i in range(1):
    plt.plot(x_plot, pdf_individual[:, i], '-', color='gray', lw=3)

plt.ylabel('Density', fontsize=fs)
plt.xticks(fontsize=fs)
plt.yticks([0, 0.5, 1, 1.5], fontsize=fs)
plt.ylim([0, 1.7])
plt.title('before\nlearning', fontsize=fs-2)

plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['bottom'].set_visible(True)
plt.gca().spines['left'].set_visible(True)

# gradual learning
savepath = outputdir +'/control_location_clusters.pkl'
savedat = load_dictionary_pickle(savepath)
E10s = savedat['E10s']
E20s = savedat['E20s']
tls = ['1-400', '400-800', '800-1400']
colors2 = ['k', colors[0], colors[4]]

for j, (E10, E20) in enumerate(zip(E10s, E20s)):
    plt.subplot(2, 4, j + 2)
   
    dat = np.array([E10.T, E20.T])
    dat = fit_cos_amplitude_along_dim(dat)
    x = dat[0, :]
    y = dat[1, :]
    data = (x - y) / (x + y)
   
    X = data.reshape(-1, 1)
    gmm = GaussianMixture(n_components=2, random_state=42)
    gmm.fit(X)
   
    sns.histplot(data, color='gray', bins=20, kde=False, stat='density', alpha=0.2, edgecolor='none')
   
    x_plot = np.linspace(data.min(), data.max(), 1000).reshape(-1, 1)
    logprob = gmm.score_samples(x_plot)
    pdf = np.exp(logprob)
    pdf_individual = gmm.predict_proba(x_plot) * pdf[:, np.newaxis] * gmm.weights_
   
    for i in range(2):
        plt.plot(x_plot, pdf_individual[:, i], '-', color=colors2[i+1], lw=3)
   
    plt.ylim([0, 1.7])
    plt.xticks([-1, 0, 1], fontsize=fs)
    plt.xlim([-1.2, 1.2])
    plt.title('trial no.\n' + tls[j], fontsize=fs-2)
    plt.yticks([], fontsize=fs)
    plt.ylabel('', fontsize=fs)
    if j==1: plt.xlabel('(g1-g2)/(g1+g2)',fontsize=fs)
   
    plt.gca().spines['left'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['bottom'].set_visible(True)

# 上半部分共享 x 标签
# fig.supxlabel('(g1-g2)/(g1+g2)', fontsize=fs, x=0.5, y=0.48)   # y值控制上下位置，可微调

# ====================== 下半部分：行为与 control state 曲线 ======================
ax1 = plt.subplot(2, 4, (5, 8))   # 占据下半部分全部4个位置

savepath = outputdir +'/control_location_trial.pkl'
savedat = load_dictionary_pickle(savepath)
E1C = savedat['E1C']
E2C = savedat['E2C']
behv1 = savedat['behv1']
behv2 = savedat['behv2']

behv = (behv1 * 1 + behv2 * 1) == 2
contl = scipy.signal.savgol_filter(np.array([E1C, E2C]).mean(0), window_length=71, polyorder=2)
contl = (contl - contl.min()) / (contl.max() - contl.min())
beh = scipy.signal.savgol_filter(behv, window_length=71, polyorder=2)
beh = (beh - beh.min()) / (beh.max() - beh.min())

corr = spearmanr(beh, contl)

# Left y-axis: gain coefficient
ax1.plot(contl, label='control state (sequence)', c=colors[3], ls='-', lw=2)
ax1.set_xlabel('trial no.', fontsize=fs)
ax1.set_ylabel('gain coefficient', fontsize=fs, color=colors[3])
ax1.tick_params(axis='x', labelsize=fs-2)
ax1.tick_params(axis='y', colors=colors[3], labelsize=fs-2)

# Right y-axis: correct rate
ax2 = ax1.twinx()
ax2.plot(beh, label='behavior', c='gray', ls='--', lw=2)
ax2.set_ylabel('correct rate', fontsize=fs, color='gray')
ax2.tick_params(axis='y', colors='gray', labelsize=fs-2)

# Spines
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax2.spines['top'].set_visible(False)
ax2.spines['left'].set_visible(False)

# Legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2,
           frameon=False, fontsize=fs-2, loc='best')

ax1.text(0.02, 0.99, f'r = {corr[0]:.2f}\np < 0.0001',
         transform=ax1.transAxes, 
         fontsize=fs-3, 
         verticalalignment='top',
         horizontalalignment='left',
         fontweight='medium',
         bbox=dict(boxstyle="round,pad=0.4", facecolor='white', alpha=0, edgecolor='none'))

# ====================== 整体布局 ======================
plt.tight_layout()
# plt.tight_layout(rect=[0, 0.03, 1, 0.96])   # 给顶部和底部留出空间
plt.show()

# fig.savefig('/mnt/data2/TP_CON/group/PFC/analysis/05_analysis/data/_figs/'+'fig6h.pdf', format='pdf', dpi=300, bbox_inches='tight')